In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import ollama  # Replaces vLLM
from ollama import AsyncClient
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# =====================================================
# INITIALIZATION & BATCH LOADING
# =====================================================

In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

In [4]:
# 3. Initialize database
print("Loading database for documents...")
vectorstore = Chroma(
    collection_name="rag_vietnamese_docs",
    persist_directory="../process_data/my_chroma_db",
    embedding_function=embeddings
)
print("Documents:", vectorstore._collection.count())

Loading database for documents...
Documents: 662071


In [5]:
# Load the Reranker Model (FlagReranker supports BGE/Qwen-based rerankers efficiently)
from FlagEmbedding import FlagReranker
print("Loading Reranker Model...")
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

Loading Reranker Model...


In [106]:
# =====================================================
# SEMANTIC ANCHORS
# =====================================================
import json
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================
# LOAD MODEL
# =====================================================
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

In [133]:
MATH_ANCHORS = [
    "query: Tính đạo hàm hàm số, tích phân, giới hạn",
    "query: Giải phương trình, tìm nghiệm biểu thức",
    "query: Bài toán xác suất, biến ngẫu nhiên, kỳ vọng, toán thống kê",
    "query: Tính vận tốc, gia tốc, động lượng, điện trở, công suất",
    "query: Tính số mol, nồng độ dung dịch, pH, hiệu suất phản ứng hóa học",
    "query: Tính doanh thu, lợi nhuận, chi phí biên, độ co giãn cung cầu",
    "query: Tính tỷ suất sinh lời, lãi suất, lãi kép, phương trình kế toán",
    "query: Tính công suất, điện trở, dòng điện"
]

# Pre-compute anchor embeddings
math_anchor_embeddings = embedding_model.encode(MATH_ANCHORS, normalize_embeddings=True)
print("Anchors and Model loaded successfully.")

# =====================================================
# HARD RULES (VIETNAMESE V3)
# =====================================================
BLACKLIST_PATTERN = re.compile(
    r"(phương trình hóa học|khái niệm|định nghĩa|lý thuyết|đặc điểm|ý nghĩa|thế nào là|phát biểu nào)", 
    re.IGNORECASE
)

MATH_PATTERN = re.compile(
    r"(đạo hàm|tích phân|giải phương trình|hệ phương trình|phương trình qd|"
    r"thể tích|bán kính|chu vi|diện tích|tốc độ thay đổi|tốc độ tăng|độ dốc|khoảng cách thẳng|khoảng cách ngang|"
    r"xác suất|phương sai|độ lệch chuẩn|"
    r"vận tốc|gia tốc|công suất|điện trở|điện dung|điện tích|hiệu điện thế|dòng điện|biến áp|nồng độ|số mol|khối lượng dung dịch|nồng độ tan|khối lượng chất|công thức hóa học|thành phần phần trăm"
    r"khúc xạ|chiết suất|góc tới|hệ số dẫn nhiệt|năng lượng liên kết|thành phần %|bao nhiêu %"
    r"doanh thu|lợi nhuận|tỷ suất|chi phí|tổng chi phí|vốn chủ sở hữu|vốn cổ phần|"
    r"độ co giãn|lãi kép|tần suất tính lãi|tỷ lệ lạm phát|số nhân tiền|chỉ số giá gdp|"
    r"hàm tổng chi phí|hàm sản xuất|sản lượng|đường cầu|lượng cầu|đường ngân sách|10 tỷ|mức giá tăng|mức giá giảm|100.000|lãi suất|khoản thu"
    r"tính \b)", 
    re.IGNORECASE
)

# =====================================================
# PHASE 1: ASYNC HYBRID ROUTER
# =====================================================
async def hybrid_router(question: str) -> tuple[str, float]:
    """
    Evaluates the question using O(1) Regex rules first, then falls back 
    to BGE-M3 semantic similarity if needed. Returns (Category, Confidence_Score).
    """
    question_lower = question.lower()

    # 1. Reading Comprehension O(1) Check
    if any(kw in question_lower for kw in ["đoạn thông tin:", "tiêu đề:", "title:", "nội dung:"]):
        return "READING", 1.0000

    # 2. LaTeX Math O(1) Check
    if re.search(r'\$.+?\$|\$\$.+?\$\$', question):
        return "MATH", 1.0000

    # 3. Vietnamese Vocabulary O(1) Check
    if not BLACKLIST_PATTERN.search(question_lower):
        if MATH_PATTERN.search(question_lower):
            return "MATH", 0.9500

    # 4. Semantic Fallback using BGE-M3 (Replacing LLM Tie-Breaker)
    q_emb = embedding_model.encode([f"query: {question}"], normalize_embeddings=True)
    similarities = cosine_similarity(q_emb, math_anchor_embeddings)[0]
    math_score = float(np.max(similarities))

    if math_score > 0.88: 
        return "MATH", math_score

    # 5. Default Fallback
    return "KNOWLEDGE", 0.9000

Anchors and Model loaded successfully.


In [137]:
import asyncio
import uuid
from FlagEmbedding import FlagReranker

# =====================================================
# PHASE 2 INITIALIZATION
# =====================================================
# Async Queue for low-confidence queries that need rewriting
async_retry_queue = asyncio.Queue()

# =====================================================
# CORE RAG FUNCTIONS
# =====================================================

async def retrieve_top_k(query: str, k: int = 5) -> list[str]:
    """
    Simulates retrieving the top 10 documents using the BGE-M3 embedding_model.
    In production, you would query your Vector DB (FAISS, Milvus, Qdrant, etc.) here.
    """
    docs = vectorstore.similarity_search(
        query,
        k=5
    )
    return docs


async def rerank_documents(query: str, docs: list[str], top_n: int = 3) -> list[tuple[str, float]]:
    """
    Uses the Cross-Encoder Reranker to score the retrieved documents against the query
    and returns the top N documents with their confidence scores.
    """
    if not docs:
        return []
        
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.compute_score(pairs)
    
    # Pair documents with their scores and sort descending
    doc_scores = list(zip(docs, scores))
    doc_scores.sort(key=lambda x: x[1], reverse=True)
    
    return doc_scores[:top_n]

async def rewrite_query(query: str) -> str:
    prompt = (
            "Bạn là một chuyên gia hỗ trợ tìm kiếm. Hãy viết lại câu hỏi dưới đây sao cho "
            "rõ ràng hơn, chi tiết hơn và tối ưu hóa cho việc tìm kiếm trong cơ sở dữ liệu.\n"
            f"Câu hỏi gốc: {query}\n"
            "Câu hỏi viết lại:"
        )
    client = AsyncClient()
    response = await client.generate(
        model='qwen2.5:7b',
        prompt=prompt,
        options={
            'temperature': 0.3,
            'num_predict': 100
        }
    )
    return response['response'].strip()

# =====================================================
# PHASE 2: PRECISION RAG PIPELINE
# =====================================================

async def phase2_precision_rag(query: str, format_answer: str = "", is_retry: bool = False):
    # 1. Retrieve Top 10 (BGE-M3)
    retrieved_docs = await retrieve_top_k(query, k=5)

    # 2. Rerank Top 3 (Cross-Encoder)
    reranked_results = await rerank_documents(query, retrieved_docs, top_n=3)

    # 3. Retrieval Confidence Check
    top_doc, top_score = reranked_results[0] if reranked_results else ("", -999)
    CONFIDENCE_THRESHOLD = 0.50

    if top_score < CONFIDENCE_THRESHOLD and not is_retry:
        print(f"⚠️ Low confidence. Moving to Async Queue for rewrite: '{query}'")
        rewritten_query = await rewrite_query(query)
        
        loop = asyncio.get_running_loop()
        reply_future = loop.create_future()
        
        # Package the format_answer into the payload
        await async_retry_queue.put({
            "original_query": query, 
            "rewritten_query": rewritten_query, 
            "format_answer": format_answer,
            "future": reply_future  
        })
        
        return "", "QUEUED_FOR_RETRY", reply_future

    final_context = "\n".join([f"- {doc.page_content}" for doc, score in reranked_results])
    return final_context, "SUCCESS", None

In [138]:
import uuid

# =====================================================
# PHASE 3: UNIFIED REASONING ENGINE (VIETNAMESE)
# =====================================================

def build_qwen_chatml_prompt(question: str, context_or_flag: str, format_answer: str) -> str:
    """
    Constructs a ChatML formatted prompt tailored for Qwen models.
    Adapts the system prompt dynamically based on whether it's Math or RAG,
    fully translated to Vietnamese.
    """
    # 1. Determine the System Instruction
    if context_or_flag == "MATH":
        system_msg = "Bạn là một chuyên gia học thuật xuất sắc trong các lĩnh vực Toán học, Vật lý, Hóa học và Kinh tế học. Hãy giải quyết bài toán từng bước một cách vô cùng logic và chính xác."        
        user_msg = f"Câu hỏi: {question}\n\n{format_answer}"
    elif context_or_flag == "READING":
        system_msg = "Bạn là một trợ lý chuyên gia về đọc hiểu. Hãy phân tích đoạn văn bản và trả lời chính xác."
        user_msg = f"Câu hỏi: {question}\n\n{format_answer}"
    else:
        system_msg = "Bạn là một trợ lý AI chuyên gia. Ưu tiên hàng đầu của bạn là trả lời câu hỏi dựa trên ngữ cảnh được cung cấp. Tuy nhiên, nếu ngữ cảnh không đủ thông tin hoặc không khớp với bất kỳ lựa chọn nào, hãy sử dụng kiến thức nền tảng của chính bạn để suy luận và chọn ra đáp án đúng nhất."
        user_msg = f"Ngữ cảnh:\n{context_or_flag}\n\nCâu hỏi: {question}\n\n{format_answer}"

    # 2. Enforce Strict Output Constraints
    format_constraint = (
            "Bạn PHẢI cấu trúc câu trả lời của mình CHÍNH XÁC bằng các thẻ XML sau. "
            "CHỈ sử dụng tiếng Việt. Tuyệt đối KHÔNG sử dụng ngôn ngữ nào khác. "
            "Không được thêm bất kỳ văn bản nào bên ngoài các thẻ này:\n"
            "<answer_reasoning>\n[Suy luận logic từng bước hoặc các bước giải toán của bạn]\n</answer_reasoning>\n"
            "<verification_check>\n[Tự đánh giá lại quá trình suy luận để tìm ra lỗ hổng hoặc sai sót tính toán]\n</verification_check>\n"
            "<final_answer>\n[CHỈ ghi MỘT chữ cái duy nhất đại diện cho đáp án đúng. Ví dụ: A, B, C, D...]\n</final_answer>"
        )

    # 3. Assemble ChatML Prompt
    prompt = (
        f"<|im_start|>system\n{system_msg}\n{format_constraint}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    return prompt

async def phase3_reasoning(question: str, context_or_flag: str, format_answer: str) -> str:
    prompt = build_qwen_chatml_prompt(question, context_or_flag, format_answer)
    
    client = AsyncClient()
    response = await client.generate(
        model='qwen2.5:7b',
        prompt=prompt,
        options={
            'temperature': 0.2,
            'top_p': 0.9,
            'num_predict': 1500,
            'stop': ["<|im_end|>"]
        }
    )
    return response['response'].strip()

In [139]:
import re
import asyncio
import pandas as pd

# =====================================================
# PHASE 4: DETERMINISTIC OUTPUT LAYER
# =====================================================

def extract_and_format_answer(raw_llm_output: str) -> str:
    match = re.search(
            r'<final_answer>\s*(.*?)(?:</final_answer>|$)',
            raw_llm_output,
            re.DOTALL | re.IGNORECASE
        )

    if not match:
        return raw_llm_output

    extracted_text = match.group(1).strip().upper()

    letter_match = re.search(r'\b([A-Z])\b', extracted_text)

    return letter_match.group(1) if letter_match else extracted_text



# def extract_and_format_answer(raw_llm_output: str) -> str:
#     """
#     Extracts the content from <final_answer> tags.
#     Returns the full string exactly as the model wrote it.
#     """
#     match = re.search(r'<final_answer>\s*(.*?)\s*</final_answer>', raw_llm_output, re.DOTALL | re.IGNORECASE)
    
#     if not match:
#         return "UNKNOWN" # Fallback if model fails to follow format
        
#     # Return the exact text, just stripping leading/trailing whitespace
#     return match.group(1).strip()

In [140]:
# import asyncio

# # =====================================================
# # PHASE 4: ORCHESTRATION & QUEUE WORKER
# # =====================================================

# async def process_user_query(user_query: str) -> str:
#     """
#     The main entry point for a user's question.
#     Routes the query, retrieves context if necessary, and generates an answer.
#     """
#     print(f"\n--- Processing Query: '{user_query}' ---")
    
#     # -----------------------------------------
#     # PHASE 1: HYBRID ROUTING
#     # -----------------------------------------
#     category, confidence = await hybrid_router(user_query)
#     print(f"🚦 Route: [{category}] (Confidence: {confidence:.4f})")

#     # -----------------------------------------
#     # PHASE 2 & 3: RAG + REASONING
#     # -----------------------------------------
#     if category in ["MATH", "READING"]:
#         # Bypass Retrieval: Send directly to Qwen with the category flag
#         print("⚡ Bypassing RAG. Sending directly to Reasoning Engine...")
#         response = await phase3_reasoning(user_query, context_or_flag=category)
#         return response

#     elif category == "KNOWLEDGE":
#         # Execute Precision RAG
#         print("🔍 Executing Precision RAG...")
#         context, status = await phase2_precision_rag(user_query)

#         if status == "SUCCESS":
#             # Confidence was high enough, generate answer based on retrieved context
#             print("✅ RAG Successful. Generating response...")
#             response = await phase3_reasoning(user_query, context_or_flag=context)
#             return response
            
#         elif status == "QUEUED_FOR_RETRY":
#             # The query was confusing and got sent to the rewrite queue
#             return (
#                 "Tôi đang phân tích chuyên sâu câu hỏi của bạn vì nó khá phức tạp. "
#                 "Hệ thống đang xử lý ngầm và sẽ trả kết quả sớm nhất."
#             )

# async def background_queue_worker():
#     """
#     A background daemon that continuously processes low-confidence 
#     queries that were rewritten and queued.
#     """
#     print("🛠️ Background Queue Worker started...")
#     while True:
#         # Wait for an item to appear in the queue
#         task = await async_retry_queue.get()
#         orig_query = task["original_query"]
#         rewritten_query = task["rewritten_query"]
        
#         print(f"\n🔄 [BACKGROUND WORKER] Retrying Query: '{orig_query}'")
#         print(f"📝 Rewritten as: '{rewritten_query}'")

#         # Rerun Phase 2 with the rewritten query (set is_retry=True to avoid infinite loops)
#         context, status = await phase2_precision_rag(rewritten_query, is_retry=True)

#         # Proceed to Phase 3 regardless of the new score, but use the best context we found
#         final_response = await phase3_reasoning(orig_query, context_or_flag=context)
        
#         print(f"✅ [BACKGROUND WORKER] Finished processing for: '{orig_query}'")
#         print(f"Result:\n{final_response}\n")

#         # Mark the task as done in the queue
#         async_retry_queue.task_done()

# # =====================================================
# # SYSTEM ENTRY POINT
# # =====================================================

# async def main():
#     # 1. Start the background worker as a concurrent task
#     worker_task = asyncio.create_task(background_queue_worker())

#     # 2. Simulate incoming user queries
#     # test_queries = [
#     #     "Tính đạo hàm của hàm số f(x) = x^2 + 3x",  # Should trigger MATH directly
#     #     "Đoạn thông tin: Anh ấy sinh năm 1990. Câu hỏi: Anh ấy sinh năm bao nhiêu?", # READ
#     #     "Bao nhiêu tuổi thì được lái xe máy ở việt nam?", # RAG (KNOWLEDGE)
#     # ]

#     test_queries = [
#         "bao nhiêu tuổi thì được lái xe ở việt nam"
#     ]

#     # Process queries concurrently
#     tasks = [process_user_query(q) for q in test_queries]
#     results = await asyncio.gather(*tasks)

#     # Print immediate results
#     for q, res in zip(test_queries, results):
#         print(f"\n[FINAL OUTPUT for '{q}']\n{res}")

#     # Wait for the background queue to finish processing any low-confidence retries
#     await async_retry_queue.join()
    
#     # Cancel the infinite background worker loop gracefully
#     worker_task.cancel()

# if __name__ == "__main__":
#     await main()

In [141]:
import asyncio

# =====================================================
# PHASE 4: ORCHESTRATION & QUEUE WORKER
# =====================================================

async def process_user_query(item: dict) -> str:

    qid = item.get("qid")
    question_text = item.get("question")
    choices = item.get("choices", [])
    correct_answer_text = item.get("answer", "").strip()

    # 2. Format the prompt with choices as bullet points
    formatted_choices = "\n".join([f"- {c}" for c in choices])
    full_query = (
        f"Các lựa chọn:\n{formatted_choices}\n\n"
        f"Yêu cầu: Hãy chọn đáp án đúng nhất. Trong thẻ <final_answer>, "
        f"hãy CHỈ viết một chữ cái duy nhất đại diện cho đáp án đúng (ví dụ: A, B, C, D, E, F, v.v.). Không giải thích hay viết thêm bất kỳ văn bản nào khác."  
    )
    
    category, confidence = await hybrid_router(question_text)

    if category in ["MATH", "READING"]:
        return await phase3_reasoning(question_text, context_or_flag=category, format_answer=full_query)

    elif category == "KNOWLEDGE":
        # Notice we are unpacking 3 variables now
        context, status, reply_future = await phase2_precision_rag(question_text, format_answer=full_query)

        if status == "SUCCESS":
            return await phase3_reasoning(question_text, context_or_flag=context, format_answer=full_query)
            
        elif status == "QUEUED_FOR_RETRY":
            # 4. Wait for the background worker to give us the answer!            
            # The code will pause here until the worker calls `reply_future.set_result()`
            final_response = await reply_future 
            return final_response

async def background_queue_worker():
    while True:
        task = await async_retry_queue.get()
        reply_future = task["future"]
        
        try:
            orig_query = task["original_query"]
            rewritten_query = task["rewritten_query"]
            full_query = task["format_answer"]

            context, status, _ = await phase2_precision_rag(rewritten_query, is_retry=True)
            final_response = await phase3_reasoning(orig_query, context_or_flag=context, format_answer=full_query)

            if not reply_future.done():
                reply_future.set_result(final_response)
                
        except Exception as e:
            # If the worker crashes, send the error back so the main loop doesn't hang forever
            if not reply_future.done():
                reply_future.set_exception(e)
        finally:
            async_retry_queue.task_done()

In [142]:
# import asyncio

# # =====================================================
# # PHASE 4: ORCHESTRATION & QUEUE WORKER
# # =====================================================

# #USE THIS TO TEST ROUTE

# async def process_user_query(item: dict) -> str:

#     qid = item.get("qid")
#     question_text = item.get("question")
#     choices = item.get("choices", [])
#     correct_answer_text = item.get("answer", "").strip()

#     # 2. Format the prompt with choices as bullet points
#     formatted_choices = "\n".join([f"- {c}" for c in choices])
#     full_query = (
#         f"Các lựa chọn:\n{formatted_choices}\n\n"
#         f"Yêu cầu: Hãy chọn đáp án đúng nhất. Trong thẻ <final_answer>, "
#         f"hãy viết LẠI CHÍNH XÁC TOÀN BỘ NỘI DUNG của đáp án bạn chọn (không viết tắt, không thêm bớt từ, không chỉnh sửa)."
#     )
    
#     category, confidence = await hybrid_router(question_text)

#     return {
#         "qid": qid,
#         "question": question_text,
#         "category": category,
#         "confidence": confidence
#     }
    
 

In [143]:
import pandas as pd
import asyncio
from tqdm import tqdm

async def run_evaluation(dataset: list, output_csv: str = "evaluation_results.csv") -> pd.DataFrame:
    """
    Evaluates a dataset of questions, expecting the model to output the full string of the answer.
    """
    results = []
    
    # 1. Start the background worker for retries
    worker_task = asyncio.create_task(background_queue_worker())
    count = 0
    for item in tqdm(dataset):
        # 3. Run through your existing pipeline
        qid = item.get("qid")
        correct_answer = item.get("answer", "").strip()
        raw_response = await process_user_query(item)
        
        # 4. Extract the full string answer using the updated Phase 4 function
        model_answer = extract_and_format_answer(raw_response)

        model_answer = model_answer.replace("$", "").strip()
        correct_answer = correct_answer.replace("$", "").strip()

        correct_answer = correct_answer.rstrip(".")
        model_answer = model_answer.rstrip(".")
        
        # 5. Evaluate correctness (ignoring case and leading/trailing spaces for safety)
        is_true = (model_answer.lower() == correct_answer.lower())
        if is_true:
            count += 1
        
        # Store results
        results.append({
            "qid": qid,
            "model_answer": model_answer,
            "correct_answer": correct_answer,
            "is_true": is_true
        })

    # 6. Clean up
    worker_task.cancel()
    
    # 7. Export to CSV
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig") 
    
    return df, count

In [144]:
# import pandas as pd
# import asyncio
# from tqdm import tqdm

# #USE THIS TO TEST ROUTE

# async def run_evaluation(dataset: list, output_csv: str = "evaluation_results.csv"):
#     """
#     Evaluates a dataset of questions, expecting the model to output the full string of the answer.
#     """
#     routing_results = []
#     for item in tqdm(dataset):
#         # 3. Run through your existing pipeline
#         raw_response = await process_user_query(item)
#         routing_results.append(raw_response)
        
#     pd.DataFrame(routing_results).to_csv(
#         "routing_results.csv",
#         index=False,
#         encoding="utf-8-sig"
#     )
    

In [ ]:
import json
import time

with open('public-test-formatted.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)

print(f"Loaded {len(test_data)} questions from public-test.json")

start_time = time.perf_counter()

results_df, count = await run_evaluation(
    test_data,
    output_csv="model_evaluation_results3.csv"
)

end_time = time.perf_counter()

print(f"Execution time: {end_time - start_time:.2f} seconds")
print(f"Accuracy: {count/len(test_data):.2f}")


In [ ]:
# correct_answer = "$ B(t) = B_0 e^{-kt} $"
# correct_answer = correct_answer.replace("$", "").strip()
# print(correct_answer)

In [ ]:
# print(await hybrid_router(
#     "Một công ty có một khoản phải thu tiền mặt trị giá 150.000 USD với lãi suất hàng năm là 4%. Khoản phải thu này sẽ đáo hạn sau 24 tháng, và năm tài chính của công ty kết thúc vào ngày 31 tháng 12. Nếu khoản phải thu được phát hành vào ngày 1 tháng 7, số tiền lãi phải thu mà công ty nên báo cáo trên bảng cân đối kế toán vào cuối năm tài chính là bao nhiêu?"
# ))

In [101]:
# import json
# import string

# def format_multiple_choice(data):
#     letters = list(string.ascii_uppercase)

#     for item in data:
#         original_choices = item.get("choices", [])
#         original_answer = item.get("answer", "")

#         new_choices = []
#         new_answer = ""

#         for index, choice in enumerate(original_choices):
#             letter = letters[index]

#             if choice == original_answer:
#                 new_answer = letter

#             new_choices.append(f"{letter}. {choice}")

#         item["choices"] = new_choices
#         item["answer"] = new_answer

#     return data


# # Read
# with open("public-test.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# # Process
# formatted_data = format_multiple_choice(data)

# # Save
# with open("public-test-formatted.json", "w", encoding="utf-8") as f:
#     json.dump(formatted_data, f, ensure_ascii=False, indent=2)

# print("Saved to public-test-formatted.json")

Saved to public-test-formatted.json
